# Lakebase Setup — Synced Tables for MMF Forecast Serving

This notebook sets up **synced tables** (reverse ETL) from the Delta forecast output
tables into the existing Lakebase Autoscaling project `many-model-forecasting-advanced`.

The Dash app in `apps/` then reads from PostgreSQL for low-latency serving
instead of hitting the SQL warehouse on every request.

**Prerequisites:** The Lakebase project already exists (UID: `297a44a4-...`).

**Steps:**
1. Verify project connectivity
2. Enable CDF on source Delta tables
3. Create synced tables
4. Create PostgreSQL indexes for app query patterns

In [0]:
PROJECT_ID = "many-model-forecasting-advanced"
BRANCH = "production"

CATALOG = "mmf_demo_aym"
SCHEMA = "m4_advanced"

SYNC_TABLES = [
    "monthly_evaluation_output",
    "monthly_scoring_output",
    "m4_monthly_train",
]

In [0]:
import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get endpoint
branch_path = f"projects/{PROJECT_ID}/branches/{BRANCH}"
endpoints = list(w.postgres.list_endpoints(parent=branch_path))
ep = endpoints[0]
host = ep.status.hosts.host
print(f"Endpoint: {ep.name}")
print(f"Host: {host}")

# Generate token and test connection
cred = w.postgres.generate_database_credential(endpoint=ep.name)
username = w.current_user.me().user_name

with psycopg.connect(
    host=host, dbname="databricks_postgres",
    user=username, password=cred.token, sslmode="require"
) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print(f"Connected: {cur.fetchone()[0]}")

In [0]:
for table in SYNC_TABLES:
    fqn = f"`{CATALOG}`.`{SCHEMA}`.`{table}`"
    spark.sql(f"ALTER TABLE {fqn} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"CDF enabled on {fqn}")

In [0]:
from databricks.sdk.service.postgres import (
    SyncedTable,
    SyncedTableSyncedTableSpec,
    SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy,
)

TABLE_PKS = {
    "monthly_evaluation_output": ["unique_id", "model", "run_id", "backtest_window_start_date"],
    "monthly_scoring_output": ["unique_id", "model", "run_id"],
    "m4_monthly_train": ["unique_id", "ds"],
}

branch_path = f"projects/{PROJECT_ID}/branches/{BRANCH}"

for table in SYNC_TABLES:
    source_fqn = f"{CATALOG}.{SCHEMA}.{table}"
    synced_table_id = f"{CATALOG}.{SCHEMA}.lb_{table}"
    pks = TABLE_PKS[table]
    try:
        op = w.postgres.create_synced_table(
            synced_table=SyncedTable(
                spec=SyncedTableSyncedTableSpec(
                    branch=branch_path,
                    source_table_full_name=source_fqn,
                    primary_key_columns=pks,
                    scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.TRIGGERED,
                    postgres_database="databricks_postgres",
                ),
            ),
            synced_table_id=synced_table_id,
        )
        print(f"Synced: {source_fqn} -> {synced_table_id}")
    except Exception as e:
        if "ALREADY_EXISTS" in str(e):
            print(f"Already exists: {synced_table_id}")
        else:
            print(f"Error: {e}")


In [0]:
# Create indexes on the synced tables for the app's common queries
index_statements = [
    # Evaluation output: frequent lookups by model + unique_id
    'CREATE INDEX IF NOT EXISTS idx_eval_model ON lb_monthly_evaluation_output (model)',
    'CREATE INDEX IF NOT EXISTS idx_eval_uid ON lb_monthly_evaluation_output (unique_id)',
    # Scoring output: frequent lookups by model + unique_id
    'CREATE INDEX IF NOT EXISTS idx_score_model ON lb_monthly_scoring_output (model)',
    'CREATE INDEX IF NOT EXISTS idx_score_uid ON lb_monthly_scoring_output (unique_id)',
    # Training data: lookups by unique_id
    'CREATE INDEX IF NOT EXISTS idx_train_uid ON lb_m4_monthly_train (unique_id)',
]

with psycopg.connect(
    host=host, dbname="databricks_postgres",
    user=username, password=cred.token, sslmode="require"
) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        for stmt in index_statements:
            try:
                cur.execute(stmt)
                print(f"OK: {stmt[:60]}...")
            except Exception as e:
                print(f"Skip: {e}")

print("\nDone! The app can now read from Lakebase.")

## Architecture

```
Delta Tables (mmf_demo_aym.m4_advanced)
  │
  ├── monthly_evaluation_output ──synced──> lb_monthly_evaluation_output
  ├── monthly_scoring_output    ──synced──> lb_monthly_scoring_output
  └── m4_monthly_train          ──synced──> lb_m4_monthly_train
                                                     │
                                    Lakebase Autoscaling (PostgreSQL 17)
                                    project: many-model-forecasting-advanced
                                                     │
                                              Dash App (apps/)
                                         low-latency OLTP reads
```

After each forecast pipeline run, trigger a sync to refresh the Lakebase tables.